# NB06 · ANN: parámetros, fidelidad y latencia

NB05 dejó una puerta de entrada que ya funciona; este notebook mide **qué paga y qué gana** al buscar de forma aproximada en vez de exacta: separar el error del índice del error del modelo, y elegir el punto de operación con una restricción declarada **antes** de ver la curva.

> 🚨 **La distinción que más nota da.**
>
> ```
> recall ANN@10  =  |IDs_ANN ∩ IDs_exactos| / 10   → ¿el ÍNDICE es fiel al espacio?
> Recall@10      =  |relevantes ∩ top10| / |rel|   → ¿la REPRESENTACIÓN es buena?
> ```
>
> Un ANN puede tener recall 1,0 y una relevancia pésima: reproduce fielmente un espacio mediocre. Y puede perder un vecino exacto sin bajar nDCG si el sustituto es igual de relevante. Por eso `aurum.ann` no reutiliza `evaluacion.recall_at_k` -mezclarlas sería aplanar justo esto-.

### Lo que se hereda y no se vuelve a decidir

| | |
|---|---|
| Motor · colección | Qdrant (R03) · `aurum_catalogo__gemini_embedding_2__A4__768` (NB04) |
| Interfaz | `BuscadorVectorial` de NB05, con un `ef` nuevo por instancia |
| Oráculo exacto | `busqueda.DenseRetriever` (ya construido y probado en NB02), no FAISS ni `SearchParams(exact=True)` |

### Las decisiones de este notebook

| | Decidido | Por qué |
|---|---|---|
| **D16** 🚨 | `recall ANN@10 ≥ 0,90` ∧ `p95 ≤ 20 ms` | Fijada **antes** de correr el barrido — si se fija después de ver la curva, deja de ser una restricción y pasa a describir el resultado. Opción "permisivo": prioriza velocidad, confiando en que la comparación nDCG lo confirme |
| **D17** | No entra el laboratorio FAISS opcional | NB06 se limita al motor de entrega; comparar HNSW/IVF/IVF-PQ queda fuera de alcance |
| `m` / `ef_construct` | Por defecto de Qdrant (**16** / **100**) | Tocarlos exige una colección nueva al lado (`HnswConfigDiff` se fija al crear). Queda anotado como mejora futura, no como decisión por omisión |
| Barrido | `ef ∈ {16, 32, 64, 128, 256}` | La tabla de familias HNSW del plan. `ef` es de **consulta**: se barre sin reconstruir nada |

> 💡 **Referencia:** `sesion_02` §7.2 aplica exactamente este procedimiento -fijar umbral, descartar lo que no llega, elegir lo más barato entre lo que sobra- y es explícito en que el umbral "no es una recomendación universal, es una decisión de producto y de riesgo".

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) + consultas de desarrollo y evaluación
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from dotenv import load_dotenv

from aurum.ann import (
    aplicar_restriccion,
    barrido_ef,
    comparar_ndcg_con_oraculo,
    comparar_ndcg_por_consulta,
    tabla_recall_por_consulta,
)
from aurum.busqueda import BuscadorVectorial, DenseRetriever, rank_queries_dense
from aurum.datos import load_csv
from aurum.embeddings import GeminiEncoder, cache_key, corpus_fingerprint, encode_corpus, truncate_dim
from aurum.evaluacion import qrels_from_judgements
from aurum.graficas import plot_ann_pareto
from aurum.motores import CATALOG_PREFIX, catalog_collection_name
from aurum.motores.qdrant import QdrantStore
from aurum.plantillas import render_template

load_dotenv(Path("..") / ".env")
DATA, CACHE = Path("..") / "data", Path("..") / "artifacts" / "embeddings"
completo = load_csv(DATA / "catalogo_productos.csv")
desarrollo = load_csv(DATA / "consultas_desarrollo.csv")
evaluacion = load_csv(DATA / "consultas_evaluacion.csv")
relevancias = load_csv(DATA / "relevancias_desarrollo.csv")

MODELO, CONTRATO, PLANTILLA = "gemini-embedding-2", "sin_contrato", "A4"
DIM, TOP_K = 768, 10
COLECCION = catalog_collection_name(model=MODELO, template=PLANTILLA, dim=DIM)

# D16, fijada en config.yaml -> nb06_ann ANTES de correr el barrido.
RECALL_MINIMO = 0.90
P95_MAXIMO_MS = 20.0
VALORES_EF = [16, 32, 64, 128, 256]

print(f"coleccion : {COLECCION}")
print(f"D16       : recall >= {RECALL_MINIMO} y p95 <= {P95_MAXIMO_MS} ms")
print(f"barrido   : ef en {VALORES_EF}")

c:\Users\asus\Master\modulos\modulo10_bbdd\practica\AURUM_MARKET\aurum-market-catalog\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


coleccion : aurum_catalogo__gemini_embedding_2__A4__768
D16       : recall >= 0.9 y p95 <= 20.0 ms
barrido   : ef en [16, 32, 64, 128, 256]


## A · El oráculo exacto, desde la caché

El oráculo son los vectores del catálogo completo con la plantilla A4 -los mismos que se ingirieron en NB04-, ya en `artifacts/embeddings/` desde entonces. Igual que en G.1 de NB04, la celda **comprueba la caché antes de llamar**: sin eso, un fallo de clave lanzaría 15.000 documentos contra la API de pago sin preguntar.

`DenseRetriever` con métrica `cosine` reproduce exactamente el ranking de Qdrant sin aproximar nada -es el mismo buscador que ya validó NB02-, así que no hace falta traer FAISS ni pedirle a Qdrant `exact=True`.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) — vectores desde la caché de NB04
CORPUS_ID = f"catalogo_productos__{PLANTILLA}"
textos_completo = render_template(completo, PLANTILLA)
clave = cache_key(
    model_id=MODELO, kind="document", contract=CONTRATO,
    corpus_id=CORPUS_ID, fingerprint=corpus_fingerprint(textos_completo),
)
if not (CACHE / f"{clave}.npy").exists():
    raise RuntimeError(
        f"Los vectores de {CORPUS_ID} no estan en cache ({clave}).\n"
        f"Codificarlos son 15.000 llamadas de pago: esta celda para en vez de "
        f"pagarlas sin avisar. Deberian estar desde NB04."
    )

_encoder = GeminiEncoder(
    api_key=os.environ.get("GEMINI_API_KEY"), model_id=MODELO,
    native_dim=3072, window=8192,
)
codificado_completo = encode_corpus(
    _encoder, textos_completo, corpus_id=CORPUS_ID,
    kind="document", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
)
vectores_completo = truncate_dim(codificado_completo.vectors, DIM)
ids_completo = completo["product_id"].tolist()

retriever_exacto = DenseRetriever(vectores_completo, ids_completo, metric="cosine")
print(f"oraculo : {vectores_completo.shape} · desde cache: {codificado_completo.stats.desde_cache}")

oraculo : (15000, 768) · desde cache: True


## B · Las consultas del barrido

Las 8 de desarrollo (`consultas_desarrollo.csv`, con juicios de relevancia) más las 12 de evaluación (`consultas_evaluacion.csv`, sin juicios) — **20 en total**, no una sola. El recall ANN no necesita etiquetas, así que entran las 20; el nDCG de la sección F solo puede medirse donde hay juicios, así que ahí solo entran las 8.

Las claves de las 8 de desarrollo son su `query_id` numérico (`"13357"`, no `"DEV-13357"`): es la clave que usan los qrels, y así el mismo diccionario sirve para el recall ANN y para el nDCG sin traducir nada.

In [ ]:
# 📄 DATOS · 📚 consultas_desarrollo.csv (8) + consultas_evaluacion.csv (12)
QUERY_IDS_DESARROLLO = [str(q) for q in desarrollo["query_id"]]
QUERY_IDS_EVALUACION = list(evaluacion["evaluation_id"])
QUERY_IDS_BARRIDO = QUERY_IDS_DESARROLLO + QUERY_IDS_EVALUACION

CONSULTAS_BARRIDO = dict(zip(QUERY_IDS_DESARROLLO, desarrollo["query_text"])) | dict(
    zip(QUERY_IDS_EVALUACION, evaluacion["query_text"])
)
QRELS = qrels_from_judgements(relevancias)

# Vectores de consulta: de la cache de NB02/NB03 (mismos corpus_id, batch)
vectores_desarrollo = encode_corpus(
    _encoder, desarrollo["query_text"].tolist(), corpus_id="consultas_desarrollo",
    kind="query", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
).vectors
vectores_evaluacion = encode_corpus(
    _encoder, evaluacion["query_text"].tolist(), corpus_id="consultas_evaluacion",
    kind="query", contract=CONTRATO, batch_size=32, cache_dir=CACHE,
).vectors
vectores_query_barrido = truncate_dim(
    np.vstack([vectores_desarrollo, vectores_evaluacion]), DIM
)

ORACULO = rank_queries_dense(
    retriever_exacto, QUERY_IDS_BARRIDO, vectores_query_barrido, k=TOP_K
)
print(f"consultas del barrido: {len(CONSULTAS_BARRIDO)} "
      f"({len(QUERY_IDS_DESARROLLO)} desarrollo + {len(QUERY_IDS_EVALUACION)} evaluación)")

consultas del barrido: 20 (8 desarrollo + 12 evaluación)


## C · El barrido de `ef`

Para cada `ef`: un `BuscadorVectorial` nuevo contra la misma colección, su recall ANN@10 frente al oráculo (media, mínimo y p5 — la media sola esconde si la pérdida se reparte o se concentra), y su latencia (calentamiento aparte, 30 repeticiones cíclicas sobre las 20 consultas).

> 💸 La primera ejecución paga la codificación de las consultas que NB05 no usó en su demo (hasta 14 llamadas nuevas, una por consulta — nunca una por `ef`, porque `codificar_consulta` cachea). Las siguientes ejecuciones no pagan ninguna.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) — la colección definitiva de NB04
from functools import lru_cache


@lru_cache(maxsize=256)
def codificar_consulta(texto: str):
    codificado = encode_corpus(
        _encoder, [texto], corpus_id="consulta_suelta",
        kind="query", contract=CONTRATO, batch_size=1, cache_dir=CACHE,
    )
    return truncate_dim(codificado.vectors, DIM)[0]


almacen = QdrantStore(
    collection=COLECCION,
    url=os.environ.get("AURUM_QDRANT_URL", "http://localhost:6333"),
    api_key=os.environ.get("AURUM_QDRANT_API_KEY"),
    prefix=CATALOG_PREFIX,
    timeout=30,
)
print(f"puntos en la coleccion: {almacen.count():,}".replace(",", ".") + 
      f" · indice al dia: {almacen.index_ready()}")


def construir_buscador(ef):
    return BuscadorVectorial(almacen, codificar_consulta, top_k=TOP_K, ef=ef)


barrido = barrido_ef(
    construir_buscador, CONSULTAS_BARRIDO, ORACULO,
    valores_ef=VALORES_EF, top_k=TOP_K, repeticiones=30, calentamiento=5,
)
barrido.style.hide(axis="index")

puntos en la coleccion: 15.000 · indice al dia: True


ef,recall_ann_at_10,recall_ann_at_10_min,recall_ann_at_10_p5,n_llamadas,ms_p50,ms_p95,ms_media,ms_min,ms_max,qps_estimado
16,0.870000,0.000000,0.285000,30,7.990000,10.300000,7.880000,5.030000,11.270000,125.100000
32,0.900000,0.000000,0.475000,30,8.510000,11.100000,8.720000,6.430000,12.980000,117.500000
64,0.940000,0.000000,0.855000,30,11.350000,20.080000,12.600000,6.420000,42.860000,88.100000
128,0.995000,0.900000,0.995000,30,9.590000,12.360000,9.840000,7.080000,12.720000,104.300000
256,0.995000,0.900000,0.995000,30,9.360000,11.580000,9.140000,5.970000,11.760000,106.800000


PRUEBA: Después de ver la gráfica, me extrañó que los tiempos de ef = 64 fueran tan altos con respecto a un ef de 128 o 256. Por eso, la siguiente celda ejecutará aisladamente las consultas con un ef de 64 y verificar si los tiempos se mantienen o es un outlier producido por la ejecución secuencial de consultas con ef inferiores que pudieran "pervertir" los tiempos

In [ ]:
# Aislado: mismo barrido_ef, solo ef=64. Separa un problema real de ef=64
# -sus tiempos salieron muy por encima de los de ef=128 y 256- de un efecto
# de arrastre por ejecutar 16→32→64→128→256 seguidos. Variable propia -no
# `barrido`- para no pisar el barrido completo que usa la sección D.
barrido_ef64_aislado = barrido_ef(
    construir_buscador, CONSULTAS_BARRIDO, ORACULO,
    valores_ef=[64], top_k=TOP_K, repeticiones=30, calentamiento=5,
)
barrido_ef64_aislado.style.hide(axis="index")

# RESULTADO: era ruido de aquella ejecución. Aislado, ef=64 da tiempos
# coherentes con lo que explora.

## D · D16 aplicada: qué configuraciones sobreviven, y cuál gana

Se descarta lo que no llega al recall mínimo o se pasa del p95 máximo; entre lo que sobra, gana el `ef` de **menor p95 medido** -el coste real, no un supuesto de que más `ef` siempre tarda más-. La tabla se enseña completa, con las dos columnas nuevas (`cumple_d16`, `elegido_r04`): así se ve también lo que no ganó, no solo el veredicto final.

In [ ]:
barrido_anotado = aplicar_restriccion(
    barrido, recall_minimo=RECALL_MINIMO, p95_maximo_ms=P95_MAXIMO_MS,
    columna_recall=f"recall_ann_at_{TOP_K}",
)

elegidas = barrido_anotado[barrido_anotado["elegido_r04"]]
if elegidas.empty:
    EF_ELEGIDO = None
    print("⚠️ Ninguna configuración cumple D16 a la vez. R04 queda sin fijar "
          "-revisar la tabla y decidir si D16 se relaja o si hace falta subir ef.")
else:
    EF_ELEGIDO = int(elegidas.iloc[0]["ef"])
    print(f"R04: ef = {EF_ELEGIDO}")

barrido_anotado.style.hide(axis="index")

R04: ef = 32


ef,recall_ann_at_10,recall_ann_at_10_min,recall_ann_at_10_p5,n_llamadas,ms_p50,ms_p95,ms_media,ms_min,ms_max,qps_estimado,cumple_d16,elegido_r04
16,0.870000,0.000000,0.285000,30,7.990000,10.300000,7.880000,5.030000,11.270000,125.100000,False,False
32,0.900000,0.000000,0.475000,30,8.510000,11.100000,8.720000,6.430000,12.980000,117.500000,True,True
64,0.940000,0.000000,0.855000,30,11.350000,20.080000,12.600000,6.420000,42.860000,88.100000,False,False
128,0.995000,0.900000,0.995000,30,9.590000,12.360000,9.840000,7.080000,12.720000,104.300000,True,False
256,0.995000,0.900000,0.995000,30,9.360000,11.580000,9.140000,5.970000,11.760000,106.800000,True,False


### D.1 · Qué consulta hay detrás de un `_min` bajo

La tabla de arriba resume, pero no dice **cuál** de las 20 consultas hunde el `_min` de una fila. Se inspeccionan **todos** los `ef` empatados en el peor `_min` del barrido -quedarse con uno solo por `idxmin()` escondería a los demás-; para mirar otro que no esté en el empate, se añade a `EFS_A_INSPECCIONAR`.

Cada bloque sale ordenado de **peor a mejor recall**, así que la consulta problemática siempre cae arriba, con su texto y los `product_id` que se perdió -no un número suelto que hay que creerse-.

In [ ]:
# Todos los ef empatados en el peor _min del barrido, no solo el primero.
MINIMO_GLOBAL = barrido_anotado[f"recall_ann_at_{TOP_K}_min"].min()
EFS_A_INSPECCIONAR = sorted(
    int(ef) for ef in barrido_anotado.loc[
        barrido_anotado[f"recall_ann_at_{TOP_K}_min"] == MINIMO_GLOBAL, "ef"
    ]
)
print(f"peor _min del barrido: {MINIMO_GLOBAL:.2f} · empatan: {EFS_A_INSPECCIONAR}\n")

bloques = []
for ef in EFS_A_INSPECCIONAR:
    buscador_inspeccion = construir_buscador(ef)
    ann_inspeccion = {
        query_id: [r.document_id for r in buscador_inspeccion.buscar(texto, top_k=TOP_K)]
        for query_id, texto in CONSULTAS_BARRIDO.items()
    }
    bloque = tabla_recall_por_consulta(
        ORACULO, ann_inspeccion, consultas=CONSULTAS_BARRIDO, k=TOP_K
    ).sort_values("recall_ann")
    bloque.insert(0, "ef", ef)
    bloques.append(bloque)

detalle_inspeccion = pd.concat(bloques, ignore_index=True)
detalle_inspeccion.style.hide(axis="index")

peor _min del barrido: 0.00 · empatan: [16, 32, 64]



ef,query_id,consulta,recall_ann,vecinos_recuperados,perdidos
16,18868,botines marrones mujer tacon medio,0.000000,0 de 10,"B0753T4R63, B07GN7XRQ9, B07H2Y8R6Y, B07H97VGBP, B07HK719R4, B07JZ62WYD, B07PL1648Y, B096VDZRTC, B09HPHX5FD, B09HT19XR9"
16,EVAL-93437-semantic,necesito un asiento cómodo para trabajar ocho horas con buen apoyo para la espalda,0.300000,3 de 10,"846086104X, B01FD2YT2E, B01N907PBW, B07BGGY1HK, B08HGBV9R4, B08ZMPDXQC, B09DVP26N3"
16,EVAL-93437-context,me duele la espalda al trabajar y necesito una silla con buen apoyo lumbar,0.700000,7 de 10,"B00DEPDZBO, B08ZMPDXQC, B09DVP26N3"
16,38249,estantes sin taladro habitacion,0.800000,8 de 10,"B01M21NLM6, B0987BCHF3"
16,33633,disfraz halloween talla grande hombre,0.800000,8 de 10,"B000PY2QWG, B07GDQ9333"
16,EVAL-100455-semantic,quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe,0.900000,9 de 10,B08F6RC895
16,EVAL-101352-semantic,busco un televisor pequeño de unas setenta centímetros para la cocina,0.900000,9 de 10,B08T7YZDQ9
16,31224,cámaras bridge baratas,1.000000,10 de 10,—
16,61533,lentejas sin gluten,1.000000,10 de 10,—
16,43240,funda ipad air 4 sin tapa,1.000000,10 de 10,—


PRUEBA: las tres configuraciones de `ef` con el `recall_ann_at_10_min` más bajo tienen el mismo problema, así que hay que verificar si es un error estructural o de densidad en las posibles respuestas — es decir, que los resultados puedan ser buenos aunque los que el oráculo daría por aptos no entren en el top 10.

In [ ]:
# ef=128 en concreto -ya no está en el empate de _min-, para ver si la
# misma consulta problemática se recupera o si aparece otra distinta.
# Variables con sufijo _128 para no pisar las de la celda D.1 de arriba.
EFS_INSPECCION_128 = [128]

bloques_128 = []
for ef in EFS_INSPECCION_128:
    buscador_inspeccion = construir_buscador(ef)
    ann_inspeccion = {
        query_id: [r.document_id for r in buscador_inspeccion.buscar(texto, top_k=TOP_K)]
        for query_id, texto in CONSULTAS_BARRIDO.items()
    }
    bloque = tabla_recall_por_consulta(
        ORACULO, ann_inspeccion, consultas=CONSULTAS_BARRIDO, k=TOP_K
    ).sort_values("recall_ann")
    bloque.insert(0, "ef", ef)
    bloques_128.append(bloque)

detalle_128 = pd.concat(bloques_128, ignore_index=True)
detalle_128.style.hide(axis="index")

# CONCLUSIÓN: es densidad de la región, no un fallo del índice. Con ef=128
# la consulta problemática recupera 10 de 10. Que ef=32 no lo haga no
# significa que sus resultados sean malos para quien busca: eso lo mide la
# comparación de nDCG contra el oráculo, más abajo.

## E · La curva recall-latencia

La región sombreada es la intersección de D16 -`p95 ≤ 20 ms` **y** `recall ≥ 0,90`-, no dos bandas cruzadas: un punto dentro de ella cumple las dos condiciones a la vez. El punto en otro color es R04.

In [ ]:
plot_ann_pareto(
    barrido_anotado, recall_minimo=RECALL_MINIMO, p95_maximo_ms=P95_MAXIMO_MS,
    recall_column=f"recall_ann_at_{TOP_K}",
    subtitle=f"{COLECCION} · {len(CONSULTAS_BARRIDO)} consultas · top_k={TOP_K}",
)

## F · Recall por consulta del `ef` elegido

El resumen (media/mínimo/p5) puede esconder si el error se reparte o se concentra; esta tabla enseña, consulta a consulta, qué `product_id` se perdió -no un booleano-.

In [ ]:
if EF_ELEGIDO is None:
    print("Sin ef elegido por D16: no hay configuración que auditar aquí.")
else:
    buscador_elegido = construir_buscador(EF_ELEGIDO)
    ann_elegido = {
        query_id: [r.document_id for r in buscador_elegido.buscar(texto, top_k=TOP_K)]
        for query_id, texto in CONSULTAS_BARRIDO.items()
    }
    detalle = tabla_recall_por_consulta(
        ORACULO, ann_elegido, consultas=CONSULTAS_BARRIDO, k=TOP_K
    )
    display(detalle.style.hide(axis="index"))

query_id,consulta,recall_ann,vecinos_recuperados,perdidos
13357,base tapizada 160x200 sin patas,1.000000,10 de 10,—
18868,botines marrones mujer tacon medio,0.000000,0 de 10,"B0753T4R63, B07GN7XRQ9, B07H2Y8R6Y, B07H97VGBP, B07HK719R4, B07JZ62WYD, B07PL1648Y, B096VDZRTC, B09HPHX5FD, B09HT19XR9"
28703,convertibles 2 en 1 portátil tactil,1.000000,10 de 10,—
31224,cámaras bridge baratas,1.000000,10 de 10,—
33633,disfraz halloween talla grande hombre,0.900000,9 de 10,B07GDQ9333
38249,estantes sin taladro habitacion,1.000000,10 de 10,—
43240,funda ipad air 4 sin tapa,1.000000,10 de 10,—
61533,lentejas sin gluten,1.000000,10 de 10,—
EVAL-100455-context,taladro sin cable de 24 voltios que venga con su batería,1.000000,10 de 10,—
EVAL-100455-direct,taladro 24v batería,1.000000,10 de 10,—


## G · La métrica clave: nDCG con el oráculo frente al ANN elegido

Si el nDCG no baja al pasar del oráculo exacto al `ef` elegido, la fidelidad perdida -si la hay- no le costó nada al negocio: es la comprobación de que optimizar por D16 (velocidad) no se hizo a costa de la calidad real. Solo entran las 8 consultas de desarrollo, las únicas con juicios de relevancia.

In [ ]:
if EF_ELEGIDO is None:
    print("Sin ef elegido por D16: no hay nDCG que comparar.")
else:
    ann_desarrollo = {qid: ann_elegido[qid] for qid in QUERY_IDS_DESARROLLO}
    oraculo_desarrollo = {qid: ORACULO[qid] for qid in QUERY_IDS_DESARROLLO}
    tabla_ndcg = comparar_ndcg_con_oraculo(
        ann_desarrollo, oraculo_desarrollo, QRELS, k=TOP_K
    )
    columnas_pct = [c for c in tabla_ndcg.columns if c != "sistema"]
    display(
        tabla_ndcg.style.hide(axis="index")
        .format({columna: "{:.1%}" for columna in columnas_pct})
    )

sistema,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10
oráculo exacto (DenseRetriever),62.5%,29.0%,93.8%,60.1%
ANN elegido (R04),60.0%,28.0%,81.2%,55.3%


### G.1 · nDCG consulta a consulta

El agregado de arriba puede esconder dos historias muy distintas: que la pérdida se reparta un poco entre las 8, o que se concentre en la misma consulta que ya delató un recall ANN bajo -la 18868, "botines marrones mujer tacon medio", si sigue siendo la peor con este `ef`-. Si su `delta` sale ~0 pese al recall ANN bajo, es la prueba de que el sustituto que trajo el ANN era igual de relevante para el negocio aunque no fuera el vecino exacto. Ordenada por `delta`: la consulta que más pierde queda arriba.

In [ ]:
if EF_ELEGIDO is None:
    print("Sin ef elegido por D16: no hay nDCG que comparar.")
else:
    ndcg_por_consulta = comparar_ndcg_por_consulta(
        ann_desarrollo, oraculo_desarrollo, QRELS,
        consultas=CONSULTAS_BARRIDO, k=TOP_K,
    )
    columnas_pct = [c for c in ndcg_por_consulta.columns if c not in ("query_id", "consulta")]
    display(
        ndcg_por_consulta.style.hide(axis="index")
        .format({columna: "{:+.1%}" if columna == "delta" else "{:.1%}" for columna in columnas_pct})
    )

query_id,consulta,ndcg_at_10_oraculo,ndcg_at_10_ann,delta
18868,botines marrones mujer tacon medio,45.0%,0.0%,-45.0%
13357,base tapizada 160x200 sin patas,69.7%,69.7%,+0.0%
28703,convertibles 2 en 1 portátil tactil,61.2%,61.2%,+0.0%
31224,cámaras bridge baratas,39.6%,39.6%,+0.0%
38249,estantes sin taladro habitacion,68.6%,68.6%,+0.0%
43240,funda ipad air 4 sin tapa,78.8%,78.8%,+0.0%
61533,lentejas sin gluten,92.1%,92.1%,+0.0%
33633,disfraz halloween talla grande hombre,25.6%,32.6%,+7.0%


## H · Lo que este notebook no mide, y por qué

| Métrica del plan | Por qué no aparece aquí |
|---|---|
| Tiempo de construcción | `m`/`ef_construct` se quedan en los valores por defecto (D16); no se reconstruye ninguna colección en este barrido, así que no hay tiempo de construcción que comparar entre configuraciones |
| Tamaño del índice | `ef` es un parámetro de **consulta**: no cambia el grafo ni el volumen en disco. El tamaño ya está medido en NB04 (289,5 MB) y es el mismo para las cinco filas de la tabla |
| Nº de distancias/consulta | El cliente de Qdrant no expone esta estadística por consulta -a diferencia de `hnsw_stats.ndis` en FAISS-. Es un caso real de "si el proveedor oculta esa decisión, explicad qué control se pierde" (§3.2): se pierde la explicación mecánica de *por qué* sube el recall al subir `ef`, aunque la curva de la sección E sí muestra *que* sube |

---

## I · El artefacto

In [ ]:
destino = Path("..") / "artifacts" / "benchmark_ann.csv"
barrido_anotado.to_csv(destino, index=False)
print(f"Escrito {destino} · {destino.stat().st_size / 1024:.1f} KB · ef elegido (R04): {EF_ELEGIDO}")

Escrito ..\artifacts\benchmark_ann.csv · 0.5 KB · ef elegido (R04): 32
